# 🧩 Notebook 3: Real-World Sidecar Patterns

In production, sidecars usually do more than just auth. This notebook walks through three patterns you'll see in the wild, plus the trade-offs of using sidecars at all.

1. **Proxy sidecar** with retries + timeouts (the Envoy / linkerd-proxy role)
2. **Log-forwarder sidecar** (the Fluent Bit / Vector role)
3. **Config-refresher sidecar** (the Vault Agent / consul-template role)

Each pattern uses only the Python standard library, so you can run everything from the notebook.

## 🛠️ Setup

```bash
cd 05-microservices/sidecar
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

### Shared helpers (idempotent server start, like Notebook 2)

In [ ]:
import threading, time, json, random, os, tempfile, urllib.request, urllib.error
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

_servers = {}
def start_server(name, port, handler_cls):
    if name in _servers:
        return _servers[name]
    srv = ThreadingHTTPServer(('127.0.0.1', port), handler_cls)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    _servers[name] = srv
    time.sleep(0.15)
    return srv

## 1) Proxy sidecar with retries + timeouts

A flaky upstream is a fact of life in distributed systems (network blips, GC pauses, cold starts). A proxy sidecar can retry transient failures **without the app doing anything**.

**Bad**: app retries in every language, each team picks different policies.
**Best**: the sidecar enforces one retry + timeout policy for the whole fleet. You can tune it centrally and get a consistent view in metrics.

### Flaky upstream app — fails ~40% of the time

In [ ]:
class FlakyApp(BaseHTTPRequestHandler):
    def do_GET(self):
        if random.random() < 0.4:  # 40% transient failure
            self.send_response(503); self.end_headers(); self.wfile.write(b'busy'); return
        body = json.dumps({'ok': True, 'path': self.path}).encode()
        self.send_response(200); self.send_header('Content-Length', str(len(body))); self.end_headers(); self.wfile.write(body)
    def log_message(self, *a, **kw): pass

start_server('flaky-app', 9101, FlakyApp)
print('flaky app on :9101')

### Retry-proxy sidecar — up to 3 attempts, 500 ms budget each

In [ ]:
class RetryProxy(BaseHTTPRequestHandler):
    UPSTREAM = 'http://127.0.0.1:9101'
    MAX_ATTEMPTS = 3
    TIMEOUT_S = 0.5

    def do_GET(self):
        last_err = None
        for attempt in range(1, self.MAX_ATTEMPTS + 1):
            try:
                with urllib.request.urlopen(f'{self.UPSTREAM}{self.path}', timeout=self.TIMEOUT_S) as r:
                    if r.status == 200:
                        body = r.read()
                        print(f'[proxy] OK on attempt {attempt}')
                        self.send_response(200); self.send_header('Content-Length', str(len(body))); self.end_headers(); self.wfile.write(body); return
                    last_err = f'HTTP {r.status}'
            except Exception as e:
                last_err = str(e)
            # exponential backoff with jitter — only retry transient failures
            time.sleep(0.05 * attempt + random.random() * 0.05)
        print(f'[proxy] gave up after {self.MAX_ATTEMPTS} attempts: {last_err}')
        self.send_response(502); self.end_headers(); self.wfile.write(b'upstream failed')
    def log_message(self, *a, **kw): pass

start_server('retry-proxy', 9100, RetryProxy)
print('retry proxy on :9100')

In [ ]:
# Drive 10 requests through the proxy. The app fails ~40% per attempt, but
# with up to 3 attempts the end-to-end success rate jumps above ~90%.
random.seed(1)
successes = 0
for _ in range(10):
    try:
        urllib.request.urlopen('http://127.0.0.1:9100/ping', timeout=2).read()
        successes += 1
    except urllib.error.HTTPError:
        pass
print(f'end-to-end success: {successes}/10')

## 2) Log-forwarder sidecar

A common sidecar job: the app writes logs to a file (or stdout), and a sidecar ships them somewhere — e.g. Elasticsearch, Loki, S3. The app doesn't know or care about the log backend.

**Bad**: app directly calls the log backend's SDK → tight coupling, language-specific libraries, hard to rotate credentials.
**Best**: app writes structured lines to a shared file; sidecar tails the file and forwards them.

In [ ]:
# The app and the sidecar share a file (in a real pod: a shared `emptyDir` volume).
log_path = os.path.join(tempfile.gettempdir(), 'sidecar_demo.log')
open(log_path, 'w').close()  # truncate on (re)run

# --- the app just appends JSON lines ---
def app_emit(event):
    with open(log_path, 'a') as f:
        f.write(json.dumps(event) + '\n')

# --- the sidecar tails the file and "ships" each line ---
shipped = []  # stands in for Elasticsearch / Loki
def log_shipper():
    with open(log_path, 'r') as f:
        f.seek(0, os.SEEK_END)  # start at end, like `tail -F`
        while True:
            line = f.readline()
            if not line:
                time.sleep(0.05)
                continue
            shipped.append(json.loads(line))

threading.Thread(target=log_shipper, daemon=True).start()

# app writes 5 events
for i in range(5):
    app_emit({'lvl': 'info', 'i': i, 'msg': 'request handled'})

time.sleep(0.3)  # let the sidecar catch up
print('shipped events:', shipped)

## 3) Config-refresh sidecar

Secrets and configs rotate. You don't want to redeploy the app every time a token changes. A sidecar can watch for new values (e.g. from Vault or a config server) and write them to a shared file the app re-reads.

**Bad**: the app embeds a secret at build time.
**Best**: the sidecar owns secret rotation; the app only reads the current value from disk.

In [ ]:
cfg_path = os.path.join(tempfile.gettempdir(), 'sidecar_token')
def atomic_write(path, content):
    # write-to-tmp + rename = readers never see a half-written file
    tmp = path + '.tmp'
    with open(tmp, 'w') as f: f.write(content)
    os.replace(tmp, path)

atomic_write(cfg_path, 'token-v1')

# --- sidecar: rotate the token every 150 ms (simulating Vault lease renewal) ---
stop_rotator = threading.Event()
def rotator():
    n = 1
    while not stop_rotator.is_set():
        n += 1
        atomic_write(cfg_path, f'token-v{n}')
        time.sleep(0.15)
threading.Thread(target=rotator, daemon=True).start()

# --- app: read the token freshly each time it needs one ---
def app_current_token():
    with open(cfg_path) as f: return f.read().strip()

seen = []
for _ in range(5):
    seen.append(app_current_token())
    time.sleep(0.18)

stop_rotator.set()
print('tokens the app observed:', seen)

## 🌍 Real-world sidecars you'll meet

| Sidecar | Role | Shipped with |
|---|---|---|
| **Envoy** | L7 proxy: mTLS, retries, traffic split, tracing | Istio, Consul Connect, AWS App Mesh |
| **linkerd-proxy** | Rust micro-proxy focused on simplicity | Linkerd |
| **Fluent Bit / Vector** | Tail container logs, ship to a backend | Most Kubernetes log stacks |
| **Vault Agent** | Fetch and rotate secrets into a shared file | HashiCorp Vault |
| **OpenTelemetry Collector** | Receive/process/export traces & metrics | OpenTelemetry |
| **cloud-sql-proxy / AlloyDB auth proxy** | Secure DB auth without embedding credentials | GCP |

## ⚖️ Trade-offs to remember

**Pros**
- App stays small and language-agnostic.
- Policies (TLS, retries, logging) are consistent across the fleet.
- Sidecars are upgraded independently of the app.

**Cons**
- Every pod pays a CPU + memory cost (×N pods). In huge clusters this adds up — it's why some teams move toward **ambient mesh** (Istio ambient, Cilium) where the mesh runs per-node instead of per-pod.
- Extra localhost hop adds latency (usually sub-millisecond, but real).
- One more moving part to operate, version, and debug (`kubectl logs pod -c istio-proxy`).
- Startup ordering matters: if the app calls out before the sidecar is ready, you get failures. Tools like Istio's `holdApplicationUntilProxyStarts` exist just to fix this.

## 🧠 Rule of thumb
> Put in the sidecar anything that is *about the network, not the domain*: TLS, retries, service discovery, telemetry, log shipping, secret refresh. Keep business logic in the app.